# Gradient descent
1. FGD
2. SGD
3. Mini-batch GD
4. SAG
# model evaluation
1. MAE
2. MSE
3. RMSE
# API
1. from sklearn.linear_model import LinearRegression
2. model = LinearRegression(fit_intercept=True)
3. from sklearn.linear_model import SGDRegressor
4. model = SGDRegressor(loss='squared_loss', learning_rate='constant', eta0=0.01, fit_intercept=True)
5. learning_rate: 'constant', 'optimal', 'invscaling', 'adaptive'
6. eta0: learning rate when learning_rate='constant'

# boston house dataset with linear regression and gradient descent demo
1. import dataset
2. extract features and target
3. split dataset into train and test
4. standardize features
5. train linear regression model with FGD
6. evaluate model with MAE, MSE, RMSE
7. SGDRegressor -> Mini-batch partial_fit

# overfit
1. L1 regularization (Lasso)
2. L2 regularization (Ridge)
3. RidgeCV for cross-validation to select best alpha

In [41]:
from tokenize import endpats

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge, Lasso, RidgeCV

import numpy as np
import pandas as pd
from win32cryptcon import X509_BASIC_CONSTRAINTS

In [27]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)

data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

In [25]:

X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)
model_linear = LinearRegression(fit_intercept=True)
model_linear.fit(X_train, y_train)
model_linear.score(X_test, y_test)
y_pred = model_linear.predict(X_test)
mae = np.mean(np.abs(y_test - y_pred))
mse = np.mean((y_test - y_pred) ** 2)
rmse = np.sqrt(mse)
print(f"MAE: {mae:.2f}, MSE: {mse:.2f}, RMSE: {rmse:.2f}")

MAE: 3.19, MSE: 24.29, RMSE: 4.93


In [30]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model_linear_scaled = LinearRegression(fit_intercept=True)
model_linear_scaled.fit(X_train_scaled, y_train)
model_linear_scaled.score(X_test_scaled, y_test)
y_pred_scaled = model_linear_scaled.predict(X_test_scaled)
mae_scaled = np.mean(np.abs(y_test - y_pred_scaled))
mse_scaled = np.mean((y_test - y_pred_scaled) ** 2)
rmse_scaled = np.sqrt(mse_scaled)
print(f"MAE: {mae_scaled:.2f}, MSE: {mse_scaled:.2f}, RMSE: {rmse_scaled:.2f}")

MAE: 3.19, MSE: 24.29, RMSE: 4.93


In [31]:
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)

model_sdg = SGDRegressor(loss='squared_error',learning_rate='constant', eta0=0.01, fit_intercept=True, max_iter=1000, tol=1e-3)
model_sdg.fit(X_train, y_train)
model_sdg.score(X_test, y_test)
y_pred_sdg = model_sdg.predict(X_test)
mae_sdg = np.mean(np.abs(y_test - y_pred_sdg))
mse_sdg = np.mean((y_test - y_pred_sdg) ** 2)
rmse_sdg = np.sqrt(mse_sdg)
print(f"SGDRegressor - MAE: {mae_sdg:.2f}, MSE: {mse_sdg:.2f}, RMSE: {rmse_sdg:.2f}")

SGDRegressor - MAE: 2090706850837045.75, MSE: 4683211208375948126525717479424.00, RMSE: 2164072828805895.00


In [32]:
scaler = StandardScaler()
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model_sdg_scaled = SGDRegressor(loss='squared_error',learning_rate='constant', eta0=0.001, fit_intercept=True, max_iter=1000, tol=1e-3)
model_sdg_scaled.fit(X_train_scaled, y_train)
model_sdg_scaled.score(X_test_scaled, y_test)
y_pred_sdg_scaled = model_sdg_scaled.predict(X_test_scaled)
mae_sdg_scaled = np.mean(np.abs(y_test - y_pred_sdg_scaled))
mse_sdg_scaled = np.mean((y_test - y_pred_sdg_scaled) ** 2)
rmse_sdg_scaled = np.sqrt(mse_sdg_scaled)
print(f"SGDRegressor with Scaling - MAE: {mae_sdg_scaled:.2f}, MSE: {mse_sdg_scaled:.2f}, RMSE: {rmse_sdg_scaled:.2f}")

SGDRegressor with Scaling - MAE: 3.22, MSE: 24.89, RMSE: 4.99


In [42]:
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 准备数据（标准化）
scaler = StandardScaler()
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 设置超参数
batch_size = 32           # training batch size
n_epochs = 1000            # number of epochs
learning_rate = 0.001


model_mb = SGDRegressor(
    loss='squared_error',
    learning_rate='constant',
    eta0=learning_rate,
    fit_intercept=True,
    random_state=42,
    warm_start=True  # 允许增量学习
)

# Mini-batch
n_samples = X_train_scaled.shape[0]
n_batches = n_samples // batch_size

for epoch in range(n_epochs):
    # shuffle training data at the beginning of each epoch
    indices = np.random.permutation(n_samples)
    X_train_shuffled = X_train_scaled[indices]
    y_train_shuffled = y_train[indices]

    # train in mini-batches
    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = start_idx + batch_size

        X_batch = X_train_shuffled[start_idx:end_idx]
        y_batch = y_train_shuffled[start_idx:end_idx]

        # use partial_fit for incremental learning
        model_mb.partial_fit(X_batch, y_batch)

    # handle the last batch if the total number of samples is not divisible by batch_size
    if n_samples % batch_size != 0:
        model_mb.partial_fit(
            X_train_shuffled[n_batches * batch_size:],
            y_train_shuffled[n_batches * batch_size:]
        )

    # evaluate training performance every 10 epochs
    if epoch % 10 == 0:
        y_pred_train = model_mb.predict(X_train_scaled)
        train_mse = np.mean((y_train - y_pred_train) ** 2)
        print(f"Epoch {epoch}, Train MSE: {train_mse:.4f}")

# evaluate on test set
y_pred_mb = model_mb.predict(X_test_scaled)
score = model_mb.score(X_test_scaled, y_test)
mae_mb = np.mean(np.abs(y_test - y_pred_mb))
mse_mb = np.mean((y_test - y_pred_mb) ** 2)
rmse_mb = np.sqrt(mse_mb)
print(("mean_squared_error:", mean_squared_error(y_test, y_pred_mb)))

print(f"\nMini-batch SGD Results:")
print(f"MAE: {mae_mb:.2f}, MSE: {mse_mb:.2f}, RMSE: {rmse_mb:.2f}")
print(f"R² Score: {model_mb.score(X_test_scaled, y_test):.4f}")


Epoch 0, Train MSE: 265.4488
Epoch 10, Train MSE: 22.4453
Epoch 20, Train MSE: 22.0382
Epoch 30, Train MSE: 21.8262
Epoch 40, Train MSE: 21.7233
Epoch 50, Train MSE: 21.7134
Epoch 60, Train MSE: 21.6789
Epoch 70, Train MSE: 21.6815
Epoch 80, Train MSE: 21.7143
Epoch 90, Train MSE: 21.6542
Epoch 100, Train MSE: 21.6498
Epoch 110, Train MSE: 21.6506
Epoch 120, Train MSE: 21.6451
Epoch 130, Train MSE: 21.6478
Epoch 140, Train MSE: 21.6560
Epoch 150, Train MSE: 21.6424
Epoch 160, Train MSE: 21.6445
Epoch 170, Train MSE: 21.6472
Epoch 180, Train MSE: 21.6496
Epoch 190, Train MSE: 21.6438
Epoch 200, Train MSE: 21.6506
Epoch 210, Train MSE: 21.6965
Epoch 220, Train MSE: 21.6487
Epoch 230, Train MSE: 21.6625
Epoch 240, Train MSE: 21.6798
Epoch 250, Train MSE: 21.6971
Epoch 260, Train MSE: 21.6498
Epoch 270, Train MSE: 21.6442
Epoch 280, Train MSE: 21.6461
Epoch 290, Train MSE: 21.6586
Epoch 300, Train MSE: 21.6783
Epoch 310, Train MSE: 21.6464
Epoch 320, Train MSE: 21.6434
Epoch 330, Train MSE

In [44]:
print(data.shape)
print(target.shape)
print(target[:10])

(506, 13)
(506,)
[24.  21.6 34.7 33.4 36.2 28.7 22.9 27.1 16.5 18.9]
